# gpu-quantum-sim · Colab T4 reproduction

A real NVIDIA GPU is required for the GPU rows. Choose **Runtime → Change runtime type → T4 GPU** before starting. We install the CUDA **12** CUDA-Q binary for broad driver compatibility. The quantum algorithms live in `src/`; this notebook only calls the CLI and shows the resulting figures.

## 1. Upload the `gpu-quantum-sim.zip` repository artifact

Download the ZIP from this conversation, then upload it when prompted. The ZIP contains this notebook and all source files. You can later replace this step with a `git clone` of your published repository.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zipfiles = [name for name in uploaded if name.endswith(".zip")]
if len(zipfiles) != 1:
    raise RuntimeError("Upload exactly one gpu-quantum-sim.zip")
with zipfile.ZipFile(zipfiles[0]) as zf:
    zf.extractall("/content")
print("Extracted project into /content/gpu-quantum-sim")

In [ ]:
%cd /content/gpu-quantum-sim
!nvidia-smi

## 2. Install pinned packages

The `cudaq` meta-package may select CUDA 13, which can need a newer driver than is present on some free Colab T4 machines. `requirements-colab.txt` uses the equivalent pinned CUDA 12 distribution. If Colab asks to restart the runtime after installing binary libraries, use **Runtime → Restart session** and continue from this cell (without re-uploading).

In [ ]:
%pip install --upgrade pip
%pip install -r requirements-colab.txt

## 3. Check the actual installed CUDA-Q API and hardware

In [ ]:
!python -m src.check_api
!python -m pytest -q

## 4. Real GHZ statevector benchmark

One warm-up and five timed runs per target per qubit count. With a typical Colab T4, n=30 requires 8 GiB FP32 state memory; CPU runs may skip larger sizes due to RAM. Any missing GPU prints an explicit reason.

In [ ]:
!python -m src.ghz_benchmark --max-qubits 30

## 5. Real H₂ quantum chemistry VQE

FCI and Hartree–Fock come independently from PySCF, and CUDA-Q computes all VQE expectation values. The full sweep has 45 bond lengths. This can take longer than the GHZ smoke test.

In [ ]:
!python -m src.h2_vqe
!python -m src.build_readme

## 6. Inspect and download real results

In [ ]:
from IPython.display import display, Image
for figure in ("ghz_benchmark.png", "ghz_speedup.png", "vqe_convergence.png", "h2_pes.png", "h2_error.png"):
    print(figure)
    display(Image(filename=f"results/{figure}"))

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("/content/gpu-quantum-sim-results", "zip", root_dir="/content/gpu-quantum-sim", base_dir="results")
files.download("/content/gpu-quantum-sim-results.zip")

### Interpretation guardrails

Do not claim an “X×” speedup without matched CPU/GPU qubit counts. The Q++ CPU is FP64 and the `nvidia` target is FP32, so the benchmark is not precision-matched. VQE is a simulator, not a quantum hardware run. A plotted error over 1.6 mHa is a failed chemical-accuracy criterion, not a success.